In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])
dbutils.widgets.dropdown("write_mode", write_mode_list[0], write_mode_list)
dbutils.widgets.dropdown("max_number_of_topics", "12", ["10", "12", "20"])

catalog = dbutils.widgets.get("source_catalog")
schema = dbutils.widgets.get("source_schema")
input_table_name = dbutils.widgets.get("input_table_name")
write_mode = dbutils.widgets.get("write_mode")
max_number_of_topics = dbutils.widgets.get("max_number_of_topics")
queue_table_name = f"{catalog}.{schema}.{refinement_queue_table_name}"

print(f"Configuration loaded:")
print(f"  Queue table: {queue_table_name}")
print(f"  Max topics: {max_number_of_topics}")

In [0]:
# ============================================================================
# CELL 3: Reset Stale Running Records
# ============================================================================
logger.info("Checking for stale 'running' records...")

stale_threshold_minutes = 15  # Consider records stale after 15 minutes

stale_records = spark.sql(f"""
    SELECT execution_id, model, topic, started_at,
           ROUND((UNIX_TIMESTAMP(CURRENT_TIMESTAMP()) - UNIX_TIMESTAMP(started_at)) / 60, 2) as minutes_running
    FROM {queue_table_name}
    WHERE status = 'running'
      AND started_at < CURRENT_TIMESTAMP() - INTERVAL {stale_threshold_minutes} MINUTES
""")

stale_count = stale_records.count()

if stale_count > 0:
    logger.warning(f"Found {stale_count} stale 'running' records (older than {stale_threshold_minutes} min)")
    stale_records.show(truncate=False)
    
    # Reset them to pending
    spark.sql(f"""
        UPDATE {queue_table_name}
        SET status = 'pending',
            started_at = NULL,
            error_message = 'Reset from stale running state (timeout)'
        WHERE status = 'running'
          AND started_at < CURRENT_TIMESTAMP() - INTERVAL {stale_threshold_minutes} MINUTES
    """)
    logger.info(f"Reset {stale_count} stale records to 'pending'")
else:
    logger.info("No stale records found")

In [0]:
if not spark.catalog.tableExists(queue_table_name):
    dbutils.notebook.exit(f"Queue table does not exist: {queue_table_name}")


In [0]:
from pyspark.sql.functions import col
import time

processed_count = 0
failed_count = 0
timeout_seconds = 600  # 10 minute timeout per notebook run

logger.info("="*60)
logger.info("Starting process_refinement_queue")
logger.info(f"Timeout per item: {timeout_seconds}s ({timeout_seconds/60:.0f} minutes)")
logger.info("="*60)

# Reset stale running records (older than 15 minutes)
logger.info("Checking for stale 'running' records...")
stale_count = spark.sql(f"""
    SELECT COUNT(*) as cnt
    FROM {queue_table_name}
    WHERE status = 'running'
      AND started_at < CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES
""").collect()[0].cnt

if stale_count > 0:
    logger.warning(f"Found {stale_count} stale 'running' records, resetting to pending...")
    spark.sql(f"""
        UPDATE {queue_table_name}
        SET status = 'pending',
            started_at = NULL,
            error_message = 'Reset from stale running state'
        WHERE status = 'running'
          AND started_at < CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES
    """)
    logger.info(f"Reset {stale_count} stale records")
else:
    logger.info("No stale records found")

# Main processing loop
while True:
    logger.info("-"*60)
    logger.info("Fetching next pending item...")
    
    pending_rows = (
        spark.table(queue_table_name)
        .filter((col("status") == "pending") & (col("layer") == 0))
        .orderBy(col("created_at").asc(), col("execution_id").asc(), col("model").asc(), col("topic").asc())
        .limit(1)
        .collect()
    )

    if not pending_rows:
        logger.info("No more pending items in queue")
        break

    row = pending_rows[0]
    logger.info(f"Processing item #{processed_count + failed_count + 1}:")
    logger.info(f"  execution_id: {row['execution_id']}")
    logger.info(f"  model: {row['model']}")
    logger.info(f"  topic: {row['topic']}")
    logger.info(f"  count: {row['count']}")

    # Update status to 'running'
    spark.sql(f"""
        UPDATE {queue_table_name}
        SET status = 'running',
            started_at = current_timestamp(),
            finished_at = NULL,
            error_message = NULL
        WHERE execution_id = '{row['execution_id']}'
          AND model = '{row['model']}'
          AND layer = {row['layer']}
          AND topic = '{row['topic']}'
          AND status = 'pending'
    """)

    start_time = time.time()
    try:
        logger.info(f"Calling ./bertopic notebook (timeout: {timeout_seconds}s)...")
        
        result = dbutils.notebook.run(
            "./bertopic",
            timeout_seconds,
            {
                "source_catalog": catalog,
                "source_schema": schema,
                "input_table_name": input_table_name,
                "write_mode": write_mode,
                "cluster_model": "KMEANS",
                "layer": str(refinement_max_layer),
                "max_number_of_topics": max_number_of_topics,
                "execution_id": row["execution_id"],
                "father_topic": str(row["topic"])
            }
        )
        
        elapsed = time.time() - start_time
        processed_count += 1
        
        logger.info(f"✓ SUCCESS in {elapsed:.1f}s ({elapsed/60:.1f} min)")
        logger.info(f"Result: {result}")
        
        # Update status to 'done'
        spark.sql(f"""
            UPDATE {queue_table_name}
            SET status = 'done',
                finished_at = current_timestamp(),
                error_message = NULL
            WHERE execution_id = '{row['execution_id']}'
              AND model = '{row['model']}'
              AND layer = {row['layer']}
              AND topic = '{row['topic']}'
        """)
        
    except Exception as e:
        elapsed = time.time() - start_time
        failed_count += 1
        
        error_type = type(e).__name__
        error_message = str(e).replace("'", "''")[:4000]
        
        logger.error(f"✗ FAILED after {elapsed:.1f}s ({elapsed/60:.1f} min)")
        logger.error(f"Error type: {error_type}")
        logger.error(f"Error: {str(e)[:500]}")
        
        # Update status to 'failed'
        spark.sql(f"""
            UPDATE {queue_table_name}
            SET status = 'failed',
                finished_at = current_timestamp(),
                error_message = '{error_message}'
            WHERE execution_id = '{row['execution_id']}'
              AND model = '{row['model']}'
              AND layer = {row['layer']}
              AND topic = '{row['topic']}'
        """)
        
        # Continue processing other items instead of stopping
        logger.warning("Continuing to next item...")
        continue

# Summary
logger.info("="*60)
logger.info("PROCESSING COMPLETE")
logger.info(f"  Succeeded: {processed_count}")
logger.info(f"  Failed: {failed_count}")
logger.info(f"  Total: {processed_count + failed_count}")
logger.info("="*60)

# Show final state
final_state = spark.table(queue_table_name).orderBy(col("created_at").asc(), col("execution_id").asc(), col("topic").asc())
display(final_state)

# Exit
dbutils.notebook.exit(f"Processed {processed_count} queued refinements ({failed_count} failed)")

In [0]:
# ============================================================================
# CELL 5: Display Results and Exit
# ============================================================================
logger.info("Fetching final queue state...")
final_state = spark.table(queue_table_name).orderBy(
    col("created_at").asc(), 
    col("execution_id").asc(), 
    col("topic").asc()
)

# Show summary statistics
summary = final_state.groupBy("status").count().orderBy("status")
logger.info("Queue status summary:")
summary.show()

# Display full table (only in interactive mode)
safe_display(final_state, "Final queue state")

# Exit with summary
exit_message = f"Processed {processed_count} queued refinements ({failed_count} failed)"
logger.info(f"Exiting: {exit_message}")
dbutils.notebook.exit(exit_message)